Cell 1 - Imports and Config

In [0]:
%python 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime 
from delta.tables import DeltaTable
import json

CATALOG = "cat_flight_liability"
BRONZE_SCHEMA = "db_bronze"
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.arrival_delays"

SILVER_SCHEMA = "db_silver"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.delay_flights_current"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")


DataFrame[]

Cell 2 - Logging

In [0]:
def log(level , message , **kwargs):
    print(json.dumps({"timestamp":datetime.utcnow().isoformat(), "level":level , "message":message , **kwargs}))

log("INFO", "Silver run started")


{"timestamp": "2026-09-01T02:34:48.015107", "level": "INFO", "message": "Silver run started"}


/home/spark-478cc391-5a96-4c0f-b069-39/.ipykernel/109/command-8098102226405003-1381834699:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp":datetime.utcnow().isoformat(), "level":level , "message":message , **kwargs}))


Cell 3 - Read Bronze

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
bronze_count = bronze_df.count()
log("INFO", "Reading bronze Data", row_count = bronze_count)

{"timestamp": "2026-09-01T02:34:49.353305", "level": "INFO", "message": "Reading bronze Data", "row_count": 0}


/home/spark-478cc391-5a96-4c0f-b069-39/.ipykernel/109/command-8098102226405003-1381834699:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp":datetime.utcnow().isoformat(), "level":level , "message":message , **kwargs}))


Cell 4 - Define the Business Key and Extract Date 

In [0]:
flight_keys_df = bronze_df.withColumn(
    "flight_date_key", F.to_date(F.col("dep_scheduled"))
)
missing_key_count = flight_keys_df.filter(
   F.col("flight_iata").isNull() | F.col("dep_scheduled").isNull()
).count()

log("INFO" , "Null keys found",total_row_count = bronze_count , null_key_count = missing_key_count)

if missing_key_count > 0:
    log("WARNING", "Some rows have no usable business key those rows will be skipped in silver")

flight_keys_df = flight_keys_df.filter(F.col("dep_scheduled").isNotNull() & F.col("flight_iata").isNotNull())

/home/spark-478cc391-5a96-4c0f-b069-39/.ipykernel/109/command-8098102226405003-1381834699:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp":datetime.utcnow().isoformat(), "level":level , "message":message , **kwargs}))


{"timestamp": "2026-09-01T02:34:50.595788", "level": "INFO", "message": "Null keys found", "total_row_count": 0, "null_key_count": 0}


Cell-5 - Build the Surrogate Key


In [0]:
silver_transformed_df = (
    flight_keys_df
    .withColumn(
    "flight_surrogate_key",
    F.sha2(F.concat_ws("|", F.col("flight_iata"), F.col("flight_date_key").cast("string")),256)
    )
    .withColumn("max_delay_minutes", F.greatest(F.col("dep_delay_minutes") , F.col("arr_delay_minutes")))
)

Cell -6 - De-Duplication 

In [0]:
# cdc happens here where where it checks the ingested at and merges the latest one this is called snapshot based cdc

window_spec = Window.partitionBy("flight_surrogate_key").orderBy(F.col("_ingested_at").desc())

latest_flight_updates_df = ( silver_transformed_df
.withColumn("row_num", F.row_number().over(window_spec))
.filter(F.col("row_num") == 1 )
.drop("row_num")
)

de_dup_count = latest_flight_updates_df.count()

log("INFO", "Deduplicated microbatch record", rows_to_merge = de_dup_count)

{"timestamp": "2026-09-01T02:34:52.227395", "level": "INFO", "message": "Deduplicated microbatch record", "rows_to_merge": 0}


/home/spark-478cc391-5a96-4c0f-b069-39/.ipykernel/109/command-8098102226405003-1381834699:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp":datetime.utcnow().isoformat(), "level":level , "message":message , **kwargs}))


Cell 7 - Merge 

In [0]:
if de_dup_count == 0 :
    log("INFO", "No data found to merge exiting")
else :
    if spark.catalog.tableExists(SILVER_TABLE):

        # the below code creates a delta table object rather than creating  delta table dataframe (spark.table(SILVER_TABLE))
        # by creating delta table it gives you access to advanced Delta Lake transactional management operations eg:- merge , upsert etc

        silver_delta_table = DeltaTable.forName(spark, SILVER_TABLE)
        
        (
            silver_delta_table.alias("target")
            .merge(
                latest_flight_updates_df.alias("source"),


                "target.flight_surrogate_key = source.flight_surrogate_key"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        merge_metrics = spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE} LIMIT 1").select("operationMetrics").collect()[0][0]
        log("INFO","Merge Metrics", updated = merge_metrics.get("numTargetRowsUpdated") , inserted = merge_metrics.get("numTargetRowsInserted"))
    else:
        latest_flight_updates_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(SILVER_TABLE)
        log("INFO", "Created initial Silver table", target=SILVER_TABLE)

log("INFO", "Silver run completed", target=SILVER_TABLE)

{"timestamp": "2026-09-01T02:34:52.687687", "level": "INFO", "message": "No data found to merge exiting"}
{"timestamp": "2026-09-01T02:34:52.687926", "level": "INFO", "message": "Silver run completed", "target": "cat_flight_liability.db_silver.delay_flights_current"}


/home/spark-478cc391-5a96-4c0f-b069-39/.ipykernel/109/command-8098102226405003-1381834699:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp":datetime.utcnow().isoformat(), "level":level , "message":message , **kwargs}))
